In [3]:
import os
import json
import pandas as pd

from timeline_features import process_match

MATCH_DIR = "data/raw/matches"
TIMELINE_DIR = "data/raw/timelines"

In [4]:
all_matches = []

match_files = [
    file
    for file in os.listdir(MATCH_DIR)
    if file.endswith(".json")
]

for i, filename in enumerate(match_files, start=1):

    match_id = filename.replace(".json", "")

    match_path = os.path.join(
        MATCH_DIR,
        filename
    )

    timeline_path = os.path.join(
        TIMELINE_DIR,
        filename
    )

    if not os.path.exists(timeline_path):
        print("Missing timeline:", match_id)
        continue

    with open(match_path) as f:
        match_data = json.load(f)

    with open(timeline_path) as f:
        timeline_data = json.load(f)

    match_df = process_match(
        match_data,
        timeline_data
    )

    all_matches.append(match_df)

    print(
        f"{i}/{len(match_files)}",
        match_id,
        len(match_df),
        "rows"
    )

1/101 NA1_5645237317 29 rows
2/101 NA1_5646584801 27 rows
3/101 NA1_5645198853 32 rows
4/101 NA1_5643969922 25 rows
5/101 NA1_5612950221 34 rows
6/101 NA1_5641543556 29 rows
7/101 NA1_5645435864 42 rows
8/101 NA1_5639870977 33 rows
9/101 NA1_5644304911 26 rows
10/101 NA1_5641037383 29 rows
11/101 NA1_5641817389 31 rows
12/101 NA1_5645523959 30 rows
13/101 NA1_5646519492 29 rows
14/101 NA1_5645436028 35 rows
15/101 NA1_5643645500 33 rows
16/101 NA1_5645374876 3 rows
17/101 NA1_5643554578 32 rows
18/101 NA1_5639899626 34 rows
19/101 NA1_5640152258 25 rows
20/101 NA1_5644327155 39 rows
21/101 NA1_5645655831 40 rows
22/101 NA1_5641372661 37 rows
23/101 NA1_5645115910 24 rows
24/101 NA1_5646599249 31 rows
25/101 NA1_5645884792 29 rows
26/101 NA1_5639554526 28 rows
27/101 NA1_5642842646 41 rows
28/101 NA1_5645631228 28 rows
29/101 NA1_5644957010 25 rows
30/101 NA1_5646501200 33 rows
31/101 NA1_5641280770 25 rows
32/101 NA1_5643940265 17 rows
33/101 NA1_5644002556 33 rows
34/101 NA1_564260541

In [5]:
df = pd.concat(
    all_matches,
    ignore_index=True
)

In [6]:
print("Rows:", len(df))
print("Matches:", df["match_id"].nunique())
print("Columns:", len(df.columns))

df.head()

Rows: 2844
Matches: 101
Columns: 37


,match_id,minute,game_version,game_duration,blue_gold,red_gold,gold_diff,blue_xp,red_xp,xp_diff,...,red_elders,blue_heralds,red_heralds,blue_barons,red_barons,blue_inhibs_alive,red_inhibs_alive,blue_nexus_turrets_alive,red_nexus_turrets_alive,blue_win
0,NA1_5645237317,0,16.18.817.5716,1623,2500,2500,0,0,0,0,...,0,0,0,0,0,3,3,2,2,0
1,NA1_5645237317,1,16.18.817.5716,1623,2515,2520,-5,0,0,0,...,0,0,0,0,0,3,3,2,2,0
2,NA1_5645237317,2,16.18.817.5716,1623,3851,3837,14,2456,2503,-47,...,0,0,0,0,0,3,3,2,2,0
3,NA1_5645237317,3,16.18.817.5716,1623,5342,5469,-127,4995,5316,-321,...,0,0,0,0,0,3,3,2,2,0
4,NA1_5645237317,4,16.18.817.5716,1623,6732,7302,-570,6711,7033,-322,...,0,0,0,0,0,3,3,2,2,0


In [7]:
print(df.isna().sum())
print(df["blue_win"].value_counts())

print(
    df.groupby("match_id")
      .size()
      .describe()
)

match_id                    0
minute                      0
game_version                0
game_duration               0
blue_gold                   0
red_gold                    0
gold_diff                   0
blue_xp                     0
red_xp                      0
xp_diff                     0
blue_cs                     0
red_cs                      0
cs_diff                     0
blue_kills                  0
red_kills                   0
kill_diff                   0
blue_towers_destroyed       0
red_towers_destroyed        0
tower_diff                  0
blue_dragons                0
red_dragons                 0
dragon_diff                 0
blue_dragons_to_soul        0
red_dragons_to_soul         0
blue_has_soul               0
red_has_soul                0
blue_elders                 0
red_elders                  0
blue_heralds                0
red_heralds                 0
blue_barons                 0
red_barons                  0
blue_inhibs_alive           0
red_inhibs

In [8]:
print(
    df["game_version"]
    .str.split(".")
    .str[:2]
    .str.join(".")
    .value_counts()
)

game_version
16.18    2810
16.15      34
Name: count, dtype: int64


In [9]:
TARGET_PATCH = "16.18"

patch = (
    df["game_version"]
    .str.split(".")
    .str[:2]
    .str.join(".")
)

df = df[patch == TARGET_PATCH].copy()

print("Rows:", len(df))
print("Matches:", df["match_id"].nunique())

print(
    df["game_version"]
    .str.split(".")
    .str[:2]
    .str.join(".")
    .value_counts()
)

Rows: 2810
Matches: 100
game_version
16.18    2810
Name: count, dtype: int64


In [11]:
import os

os.makedirs("data/processed", exist_ok=True)

df.to_csv(
    "data/processed/timeline_features_16_18_master.csv",
    index=False
)

df.to_parquet(
    "data/processed/timeline_features_16_18_master.parquet",
    index=False
)